<a href="https://colab.research.google.com/github/Zahra-Alikhani2004/ML_Final-project-/blob/main/Ml_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ====== نصب پیش‌نیازها ======
!pip install --quiet streamlit pyngrok opencv-python-headless matplotlib tensorflow scikit-learn openpifpaf==0.13.1

# ====== نوشتن فایل Streamlit ======
%%writefile app.py
import streamlit as st
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tempfile import NamedTemporaryFile
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Flatten, Concatenate
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler
import openpifpaf

st.set_page_config(page_title='⚽ استعداد‌یابی فوتسال با Pose Detection', layout='wide')
st.title('⚽ استعداد‌یابی فوتسال با Pose Detection')

# ===== Sidebar: اطلاعات بازیکن =====
st.sidebar.header("اطلاعات بازیکن")
height = st.sidebar.number_input("قد (متر)", 1.4, 2.2, 1.75)
weight = st.sidebar.number_input("وزن (کیلوگرم)", 40, 120, 70)
start_age = st.sidebar.number_input("سن شروع حرفه‌ای", 10, 40, 18)
speed = st.sidebar.slider("سرعت (1-10)", 1, 10, 7)
agility = st.sidebar.slider("چابکی (1-10)", 1, 10, 7)
endurance = st.sidebar.slider("استقامت (1-10)", 1, 10, 7)
passing = st.sidebar.slider("پاس دادن (1-10)", 1, 10, 7)
dribbling = st.sidebar.slider("دریبل (1-10)", 1, 10, 7)
shooting = st.sidebar.slider("شوت زنی (1-10)", 1, 10, 7)

# ===== Main: آپلود تصویر/ویدیو =====
st.header("آپلود عکس یا ویدیو بازیکن (اختیاری)")
uploaded_file = st.file_uploader("انتخاب فایل", type=['jpg','jpeg','png','mp4'])
img_ready = None
poses = None

if uploaded_file:
    tfile = NamedTemporaryFile(delete=False)
    tfile.write(uploaded_file.read())
    file_path = tfile.name

    if uploaded_file.type.startswith('video'):
        st.video(file_path)
        cap = cv2.VideoCapture(file_path)
        frames = []
        for _ in range(5):  # نمونه 5 فریم اول
            ret, frame = cap.read()
            if not ret: break
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        cap.release()
        if frames:
            frames_resized = [cv2.resize(f, (224,224)) for f in frames]
            img_ready = np.array(frames_resized)/255.0
            frame_for_pose = frames[0]
    else:
        img = cv2.imread(file_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_ready = np.expand_dims(cv2.resize(img_rgb, (224,224))/255.0, axis=0)
        frame_for_pose = img_rgb
        st.image(img_rgb, caption="عکس آپلود شده", use_column_width=True)

    # ===== Pose Detection با OpenPifPaf =====
    predictor = openpifpaf.Predictor(checkpoint='resnet50')
    predictions, _, _ = predictor.numpy_image(frame_for_pose)
    poses = predictions

# ===== تحلیل بازیکن =====
if st.button("تحلیل بازیکن"):
    st.subheader("در حال تحلیل بازیکن...")

    # استخراج ویژگی‌های تصویر با MobileNetV2
    if img_ready is not None:
        base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
        x = base_model.output
        x = Flatten()(x)
        image_model = Model(inputs=base_model.input, outputs=x)
        if len(img_ready.shape) == 4 and img_ready.shape[0] > 1:  # ویدیو
            image_features = np.mean(image_model.predict(img_ready), axis=0).reshape(1,-1)
        else:
            image_features = image_model.predict(img_ready)
    else:
        image_features = np.zeros((1,7*7*1280))  # placeholder

    # ویژگی‌های متنی
    user_vector = np.array([[height, weight, start_age, speed, agility, endurance, passing, dribbling, shooting]])
    scaler = MinMaxScaler()
    user_text_scaled = scaler.fit_transform(user_vector)

    # ترکیب ویژگی‌ها
    input_text = Input(shape=(user_text_scaled.shape[1],))
    input_image = Input(shape=(image_features.shape[1],))
    combined = Concatenate()([input_text, input_image])

    x = Dense(128, activation='relu')(combined)
    x = Dense(64, activation='relu')(x)
    output = Dense(6, activation='softmax')(x)
    final_model = Model(inputs=[input_text, input_image], outputs=output)
    final_model.compile(optimizer=Adam(0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # آموزش نمایشی (dummy)
    X_text_scaled = np.repeat(user_text_scaled, 6, axis=0)
    X_image_features = np.repeat(image_features, 6, axis=0)
    y_dummy = np.arange(6)
    final_model.fit([X_text_scaled, X_image_features], y_dummy, epochs=3, verbose=0)

    # پیش‌بینی
    pred = final_model.predict([user_text_scaled, image_features])
    pred_class = np.argmax(pred)
    positions = ["دروازه‌بان","مدافع","هافبک","وینگر","مهاجم","Pivot"]

    # ارزیابی مناسب بودن بازیکن
    MIN_HEIGHT, MAX_HEIGHT = 1.60, 2.00
    skills_avg = np.mean([speed, agility, endurance, passing, dribbling, shooting])
    if MIN_HEIGHT <= height <= MAX_HEIGHT and 18 <= start_age <= 30 and skills_avg >= 6:
        st.success("بازیکن برای فوتسال مناسب است ⚽")
        st.info(f"پوزیشن پیشنهادی: {positions[pred_class]}")
    else:
        st.error("بازیکن برای فوتسال مناسب نیست ❌")

    # نمودار رادار مهارت‌ها
    skills = [speed, agility, endurance, passing, dribbling, shooting]
    skill_labels = ["سرعت","چابکی","استقامت","پاس","دریبل","شوت"]
    angles = np.linspace(0, 2*np.pi, len(skills), endpoint=False).tolist()
    skills += skills[:1]
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(5,5), subplot_kw=dict(polar=True))
    ax.plot(angles, skills, 'o-', linewidth=2, label="مهارت‌های بازیکن")
    ax.fill(angles, skills, alpha=0.25)
    ax.set_thetagrids(np.degrees(angles[:-1]), skill_labels)
    ax.set_ylim(0,10)
    st.pyplot(fig)

    # نمایش Pose Detection
    if poses:
        st.subheader("Pose Detection")
        for person in poses:
            for keypoint in person.data:
                x, y, c = keypoint
                cv2.circle(frame_for_pose, (int(x), int(y)), 4, (255,0,0), -1)
        st.image(frame_for_pose, caption="Pose Detection", use_column_width=True)

# ===== اجرا با ngrok =====
from pyngrok import ngrok
import subprocess
import time

subprocess.Popen(["streamlit", "run", "app.py", "--server.port=8501"])
time.sleep(5)
public_url = ngrok.connect(8501)
print("لینک Streamlit:", public_url)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.0/224.0 kB 5.3 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Installing build dependencies ... error
error: subprocess-exited-with-error

× pip subprocess to install build dependencies did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


UsageError: Line magic function `%%writefile` not found.
